# Guardiã AI — executar no Colab (um notebook)

Substitui a corrida 01→10 só para **subir a UI**. Runtime → GPU (A100/L4 se for usar Llama).

**Run all** (Ambiente de execução → Executar tudo). No final aparece um link Gradio (`share=True`, ~72 h).

| O que precisa | Para quê |
|---|---|
| Este notebook no Colab | orquestração |
| GPU + `HF_TOKEN` (Secrets do Colab ou `AssistenteHospitalar/.env`) | Llama 3.2 3B (respostas reais no chat) |
| Sem GPU / sem token | sobe igual, com `FakeChatModel` (textos repetidos) |

Não retreina QLoRA. Chroma só entra se `files/chroma/` já existir no Drive; senão usa o RAG lexical do repositório.

Se aparecer `RuntimeError: Only a single TORCH_LIBRARY`: **Ambiente de execução → Reiniciar sessão** e rode de novo (não importe `torch` antes do `pip`).

In [ ]:
# 1) Dependências — não importe torch nesta célula (erro TORCH_LIBRARY no Colab 3.13)
import shutil, subprocess
GPU = shutil.which('nvidia-smi') is not None and subprocess.call(
    ['nvidia-smi'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
) == 0
print('GPU:', 'nvidia-smi ok' if GPU else 'nenhuma — LLM fake')

%pip install -q faker python-dotenv pydantic pytest gradio==5.29.0 \
    langchain-core==0.3.59 langgraph==0.4.5 langchain langchain-community \
    langchain-huggingface numpy pandas pyarrow scikit-learn joblib

if GPU:
    # sem -U para não substituir o torch pré-instalado do Colab
    %pip install -q transformers peft accelerate bitsandbytes \
        chromadb sentence-transformers

In [ ]:
# 2) Drive + código do GitHub
import os, sys, subprocess
from pathlib import Path
from google.colab import drive
from dotenv import load_dotenv

drive.mount('/content/drive', force_remount=True)

REPO_URL = 'https://github.com/Evandrorsampaio/TechChallenger4.git'
REPO_DIR = Path('/content/TechChallenger4')
DRIVE_BASE = Path('/content/drive/MyDrive/AssistenteHospitalar')
DRIVE_BASE.mkdir(parents=True, exist_ok=True)
(DRIVE_BASE / 'files').mkdir(exist_ok=True)

if not (REPO_DIR / 'lib').exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
else:
    subprocess.call(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'])

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

os.environ['DRIVE_BASE'] = str(DRIVE_BASE)
os.environ['HOSPITAL_DB_PATH'] = str(DRIVE_BASE / 'files' / 'hospital.db')
os.environ['ARTIFACTS_DIR'] = str(REPO_DIR / 'artifacts')
os.environ['ML_RISCO_HABILITADO'] = '1'
os.environ.setdefault('PERFIL_EXECUCAO', 'full-gpu' if GPU else 'demo-cpu')

load_dotenv(DRIVE_BASE / '.env')
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
    if tok:
        os.environ['HF_TOKEN'] = tok
except Exception:
    pass

print('REPO', REPO_DIR)
print('DB  ', os.environ['HOSPITAL_DB_PATH'])

In [ ]:
# 3) Hugging Face (só se houver token)
HF = os.environ.get('HF_TOKEN')
if HF:
    from huggingface_hub import login
    login(token=HF)
    print('HF_TOKEN ok')
else:
    print('Sem HF_TOKEN — chat usará FakeChatModel. Colab → Secrets → HF_TOKEN, ou Drive/.env')

In [ ]:
# 4) Banco mock (cria se estiver vazio)
from lib.db import connect, init_schema
from lib.mock_data import populate

conn = connect()
init_schema(conn)
n = conn.execute('SELECT COUNT(*) AS c FROM pacientes').fetchone()['c']
if n == 0:
    populate(conn, n_pacientes=50, seed=42, verbose=False)
    n = conn.execute('SELECT COUNT(*) AS c FROM pacientes').fetchone()['c']
print('pacientes', n)

In [ ]:
# 5) Modelos de risco gestacional (sklearn; não é Llama)
joblibs = list((REPO_DIR / 'artifacts' / 'models').rglob('*.joblib'))
if not joblibs:
    print('Sem .joblib no clone (gitignore). Treinando no Colab…')
    subprocess.check_call([sys.executable, 'scripts/train.py'])
else:
    print('modelos já presentes:', len(joblibs))
    subprocess.check_call([sys.executable, 'scripts/train.py', '--verificar-dataset'])

In [ ]:
# 6) RAG: Chroma no Drive se existir; senão subset lexical do repo
CHROMA_DIR = DRIVE_BASE / 'files' / 'chroma'
COLLECTION = 'protocolos_saude_mulher'

chroma_ok = CHROMA_DIR.exists() and any(CHROMA_DIR.iterdir())
if chroma_ok and GPU:
    from langchain_huggingface import HuggingFaceEmbeddings
    from langchain_community.vectorstores import Chroma
    embeddings = HuggingFaceEmbeddings(
        model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2',
        model_kwargs={'device': 'cuda'},
        encode_kwargs={'normalize_embeddings': True},
    )
    vectorstore = Chroma(
        collection_name=COLLECTION,
        embedding_function=embeddings,
        persist_directory=str(CHROMA_DIR),
    )
    retriever = vectorstore.as_retriever(search_kwargs={'k': 4})
    print('RAG Chroma:', vectorstore._collection.count(), 'chunks')
else:
    from lib.rag_local import build_local_retriever
    retriever = build_local_retriever()
    print('RAG lexical (artifacts/rag). Para Chroma: rode 06_indexar_protocolos.ipynb uma vez.')

In [ ]:
# 7) LLM: Llama+QLoRA se GPU+token; senão fake
chat_model = None
if GPU and HF:
    try:
        from lib.llm import load_finetuned, build_chat_model
        model, tokenizer = load_finetuned()
        chat_model = build_chat_model(model, tokenizer)
        print('LLM Llama pronto')
    except Exception as exc:
        print('Falha ao carregar Llama:', type(exc).__name__, exc)
if chat_model is None:
    from lib.llm_fake import FakeChatModel
    chat_model = FakeChatModel()
    print('Usando FakeChatModel')

In [ ]:
# 8) Agente + 5 workflows (inclui risco_ml)
from lib.agent import build_agent
from lib.tools import build_langchain_tools
from lib.workflows import (
    build_triagem_workflow, build_violencia_workflow,
    build_obstetrico_workflow, build_prevencao_workflow,
    build_risco_ml_workflow,
)

tools_list = build_langchain_tools(conn, retriever)
try:
    agent = build_agent(chat_model, tools_list)
except Exception as exc:
    print('Agente ReAct indisponível:', exc)
    agent = None

workflows = {
    'triagem': build_triagem_workflow(chat_model, conn, retriever),
    'violencia': build_violencia_workflow(chat_model, conn, retriever),
    'obstetrico': build_obstetrico_workflow(chat_model, conn, retriever),
    'prevencao': build_prevencao_workflow(chat_model, conn, retriever),
    'risco_ml': build_risco_ml_workflow(chat_model, conn, retriever),
}
print('tools', [t.name for t in tools_list])
print('workflows', list(workflows))

In [ ]:
# 9) Gradio — rode no navegador Colab (não no túnel VS Code)
from lib.ui import build_ui

app = build_ui(agent, conn, default_usuario='dr_residente_demo', workflows=workflows)
app.launch(share=True, debug=False, show_error=True)

Para encerrar: `app.close(); conn.close()`.

QLoRA / extração de 39 PDFs continuam nos notebooks `02` e `03`. Este arquivo só executa a demo.